# Python 비동기 기초

### 비동기란?

프로그램에는 네트워크 응답, 데이터베이스 조회, 파일 읽기처럼 결과가 올 때까지 기다려야 하는 작업이 많습니다.

**동기 방식**은 한 작업이 끝날 때까지 기다린 후 다음 작업을 시작합니다. 반면 **비동기 방식**은 한 작업이 기다리는 동안 실행할 수 있는 다른 작업을 진행합니다.

| 방식 | 작업이 기다릴 때 | 예시 흐름 |
|---|---|---|
| 동기 | 프로그램도 함께 기다림 | A 시작 → A 완료 → B 시작 → B 완료 |
| 비동기 | 다른 작업을 진행함 | A 요청 → 기다리는 동안 B 요청 → 완료된 작업부터 이어서 처리 |

비동기는 작업 자체의 계산보다 결과를 기다리는 시간이 긴 경우에 사용합니다. 네트워크 API 요청, 데이터베이스 조회, 파일 읽기와 쓰기처럼 입출력 대기가 많은 작업을 여러 개 처리할 때 적합합니다.

### 이벤트 루프

Python에서는 **이벤트 루프**가 실행할 코루틴을 관리합니다. 코루틴이 `await`에서 기다리기 시작하면 이벤트 루프는 그동안 실행할 수 있는 다른 코루틴을 찾습니다. 기다리던 결과가 도착하면 중단했던 위치부터 다시 실행합니다.

`작업 A 요청` → `A가 응답을 기다림` → `그동안 작업 B 진행` → `A의 응답이 오면 A를 이어서 실행`

단, 실행할 코루틴이 이벤트 루프에 여러 개 등록되어 있어야 기다리는 동안 다른 작업을 진행할 수 있습니다. 코루틴을 하나씩 `await`하면 각 작업이 끝난 뒤 다음 작업을 시작합니다.

### 동기 실행

동기 코드는 앞 작업이 끝난 다음에 다음 작업을 시작합니다. `time.sleep()`으로 시간이 걸리는 작업을 흉내 냅니다.

In [1]:
import time


def sync_work(name: str, seconds: int) -> str:
    print(name, "시작")
    time.sleep(seconds)
    print(name, "완료")
    return f"{name} 결과"


start = time.perf_counter()
sync_results = []
for name in ["작업 A", "작업 B", "작업 C"]:
    result = sync_work(name, 1)
    sync_results.append(result)

print("결과:", sync_results)
print(f"걸린 시간: {time.perf_counter() - start:.2f}초")

작업 A 시작
작업 A 완료
작업 B 시작
작업 B 완료
작업 C 시작
작업 C 완료
결과: ['작업 A 결과', '작업 B 결과', '작업 C 결과']
걸린 시간: 3.00초


### 코루틴과 `async`/`await`

`async def`로 정의한 함수를 코루틴 함수라고 합니다. 코루틴 함수를 호출하면 바로 최종 결과가 나오는 것이 아니라 코루틴 객체가 만들어집니다. 이 객체를 `await`하면 실행이 끝날 때까지 기다린 뒤 결과를 받습니다.

`asyncio.sleep()`은 기다리는 동안 이벤트 루프가 다른 코루틴을 실행할 수 있게 합니다. 하지만 아래처럼 코루틴을 하나씩 `await`하면 순서대로 실행되므로, `async`를 사용했다고 자동으로 여러 작업이 함께 실행되는 것은 아닙니다.

In [2]:
import asyncio


async def async_work(name: str, seconds: int) -> str:
    print(name, "시작")
    await asyncio.sleep(seconds)
    print(name, "완료")
    return f"{name} 결과"


start = time.perf_counter()
async_results = []
for name in ["작업 A", "작업 B", "작업 C"]:
    result = await async_work(name, 1)
    async_results.append(result)

print("결과:", async_results)
print(f"걸린 시간: {time.perf_counter() - start:.2f}초")

작업 A 시작
작업 A 완료
작업 B 시작
작업 B 완료
작업 C 시작
작업 C 완료
결과: ['작업 A 결과', '작업 B 결과', '작업 C 결과']
걸린 시간: 3.04초


### `asyncio.gather()`로 함께 실행하기

`asyncio.gather()`에 여러 코루틴을 전달하면 한 작업이 기다리는 동안 다른 작업을 진행합니다. 반환되는 결과의 순서는 작업이 끝난 순서가 아니라 `gather()`에 전달한 순서와 같습니다.

In [8]:
start = time.perf_counter()

async_results = await asyncio.gather(
    async_work("작업 A", 1),
    async_work("작업 B", 1),
    async_work("작업 C", 1),
)

print("결과:", async_results)
print(f"걸린 시간: {time.perf_counter() - start:.2f}초")

작업 A 시작
작업 B 시작
작업 C 시작
작업 A 완료
작업 B 완료
작업 C 완료
결과: ['작업 A 결과', '작업 B 결과', '작업 C 결과']
걸린 시간: 1.01초


### API 요청: 하나씩 보내기 vs 비동기로 묶어서 보내기

JSONPlaceholder의 게시글 API를 5번 요청하여 순차 실행과 `asyncio.gather()` 실행 시간을 비교합니다. 네트워크 상태에 따라 측정값은 달라질 수 있습니다.

`httpx`는 `requests`와 사용법이 비슷하면서 비동기 요청을 지원하는 HTTP 클라이언트 라이브러리입니다. `pip install httpx`로 설치합니다.

In [4]:
import httpx


async def get_post(client: httpx.AsyncClient, post_id: int) -> dict:
    url = f"https://jsonplaceholder.typicode.com/posts/{post_id}"
    response = await client.get(url)
    response.raise_for_status()
    return response.json()


post_ids = [1, 2, 3, 4, 5]

### 요청 하나씩 보내기

In [5]:
async with httpx.AsyncClient(timeout=10.0) as http_client:
    start = time.perf_counter()
    sequential_posts = []
    for post_id in post_ids:
        post = await get_post(http_client, post_id)
        sequential_posts.append(post)
    sequential_time = time.perf_counter() - start

print(f"걸린 시간: {sequential_time:.2f}초")

걸린 시간: 0.81초


### 요청 함께 보내기

In [6]:
async with httpx.AsyncClient(timeout=10.0) as http_client:
    start = time.perf_counter()
    concurrent_posts = await asyncio.gather(
        *(get_post(http_client, post_id) for post_id in post_ids)
    )
    concurrent_time = time.perf_counter() - start

saved_time = sequential_time - concurrent_time
speedup = sequential_time / concurrent_time if concurrent_time else float("inf")

print(f"걸린 시간: {concurrent_time:.2f}초")
print(f"줄어든 시간: {saved_time:.2f}초")
print(f"속도 배율: {speedup:.2f}배")
print("\n응답 결과:")
for post in concurrent_posts:
    print(post["id"], post["title"])

걸린 시간: 0.30초
줄어든 시간: 0.51초
속도 배율: 2.68배

응답 결과:
1 sunt aut facere repellat provident occaecati excepturi optio reprehenderit
2 qui est esse
3 ea molestias quasi exercitationem repellat qui ipsa sit aut
4 eum et est occaecati
5 nesciunt quas odio


### 실습 문제

JSONPlaceholder의 사용자 API를 비동기로 요청하세요.

- 주소는 `https://jsonplaceholder.typicode.com/users/{user_id}`입니다.
- `get_user(client, user_id)` 코루틴 함수를 작성합니다.
- 하나의 `httpx.AsyncClient`를 세 요청에서 함께 사용합니다.
- 사용자 ID 1, 2, 3을 `asyncio.gather()`로 요청합니다.
- 각 사용자의 `id`, `name`, `email`을 출력합니다.
- 전체 실행 시간을 출력합니다.

In [ ]:
# TODO: get_user 코루틴 함수를 작성하세요.
async def get_user():
    return

# TODO: 사용자 3명을 함께 요청하고 결과와 실행 시간을 출력하세요.
async with httpx.AsyncClient() as http_client:
    result = await asyncio.gather(
    get_user(), get_user(), get_user()
)

### Jupyter Notebook과 Python 파일에서 실행하기

Jupyter Notebook과 Colab은 이미 이벤트 루프가 실행 중이므로 셀에서 `await`를 바로 사용할 수 있습니다.

```python
result = await async_work("작업", 1)
```

일반 `.py` 파일에서는 최상위 코드에 `await`를 바로 쓸 수 없습니다. 아래 코드를 `.py` 파일로 저장한 뒤 실행합니다. Notebook 안에서 `asyncio.run()`을 호출하면 이미 실행 중인 이벤트 루프와 충돌할 수 있습니다.

In [7]:
import asyncio


async def async_work(name: str, seconds: int) -> str:
    print(name, "시작")
    await asyncio.sleep(seconds)
    print(name, "완료")
    return f"{name} 결과"


async def main():
    results = await asyncio.gather(
        async_work("작업 A", 1),
        async_work("작업 B", 1),
        async_work("작업 C", 1),
    )
    print("결과:", results)


if __name__ == "__main__":
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop